# Waymaker × Gemma 4 / Qwen — LoRA/QLoRA fine-tuning experiment (Google Colab)

**Experimental only. Not wired into the live Waymaker backend. Not production-ready.**

This notebook fine-tunes a small instruction model with QLoRA for **behavior and
output formatting only**:

- answer **only** from the `evidence_pack` in the user message
- refuse / defer when evidence is insufficient
- never invent fees, periods, document lists, or eligibility rules
- produce a consistent 6-section Korean guidance structure
- preserve source grounding (cite the sources used)

It must **not** teach the model visa rules, fees, periods, eligibility requirements,
or document lists. The bundled sample data uses clearly-marked fake placeholder
excerpts — no real legal facts.

## Designed to run on **Google Colab Free** (T4 16 GB)

The default preset is a **small open Qwen model** with an **ultra-low-memory** smoke
config, so the whole pipeline (load → train → save → infer → score) runs end-to-end on
a free T4 without a Hugging Face token. Pick a model with the preset selector in the
**config cell**:

| Preset | Model | Free Colab (T4)? | Token needed? | Role |
|---|---|---|---|---|
| `qwen-0.6b` | `Qwen/Qwen3-0.6B` | ✅ safest | no | smallest smoke test |
| `qwen-1.7b` *(default)* | `Qwen/Qwen3-1.7B` | ✅ good default | no | smoke test |
| `qwen-4b` | `Qwen/Qwen3-4B` | ⚠️ only if GPU has spare memory | no | larger smoke test |
| `gemma-e4b` | `google/gemma-4-E4B-it` | ⚠️ **may still OOM** on free Colab | **yes (gated)** | original Gemma smoke path |
| `gemma-12b` | `google/gemma-4-12B-it` | ❌ needs Colab Pro L4/A100 | **yes (gated)** | Stage 2 quality run |

**Smoke test vs quality:** every preset here except `gemma-12b` is a **wiring smoke
test** — it only proves the dataset format, chat template, LoRA/QLoRA code, adapter
save/load, inference, and eval script work. Answer quality is **not** a signal on the
small models. The actual Waymaker quality experiment is still `gemma-12b` on Colab Pro.

**Runtime:** a GPU runtime is required (Runtime → Change runtime type → **T4 GPU**).
The first code cell checks this for you.

Training is on **final visible assistant answers only** — no hidden chain-of-thought
is included in the data or the loss.

In [ ]:
# 0) GPU + memory check  —  RUN THIS FIRST
# Free Colab usually assigns a T4 (16 GB), but sometimes assigns NO GPU at all.
# This cell tells you which case you're in, recommends a model preset, and — if no
# GPU is assigned — prints the exact fallback steps to get one.
import subprocess

HAS_GPU = False
GPU_NAME = ""
GPU_TOTAL_GB = 0.0
GPU_FREE_GB = 0.0
try:
    import torch  # pre-installed on Colab
    HAS_GPU = torch.cuda.is_available()
except Exception as e:
    print("Could not import torch yet:", e)

if HAS_GPU:
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free_b, total_b = torch.cuda.mem_get_info()
    GPU_FREE_GB = free_b / 1024**3
    print(f"✅ GPU assigned: {GPU_NAME}")
    print(f"   total memory: {GPU_TOTAL_GB:.1f} GB | free right now: {GPU_FREE_GB:.1f} GB")
    try:
        print("\n" + subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free",
             "--format=csv"], capture_output=True, text=True).stdout)
    except Exception:
        pass

    # Recommend a preset based on total memory.
    if GPU_TOTAL_GB >= 22:
        print("Plenty of memory. You can try 'qwen-4b', or 'gemma-12b' on L4/A100 (Stage 2).")
    elif GPU_TOTAL_GB >= 15:
        print("Looks like a free-tier T4 (~16 GB).")
        print("Recommended: keep MODEL_CHOICE='qwen-1.7b' (or 'qwen-0.6b') with ULTRA_LOW_MEM=True.")
        print("'qwen-4b' MIGHT fit at max_seq_length=512; 'gemma-e4b' MAY still OOM here.")
    else:
        print("Small GPU detected. Use 'qwen-0.6b' with ULTRA_LOW_MEM=True.")
else:
    # ---- FALLBACK PATH: no GPU assigned ----
    print("❌ No GPU is assigned to this runtime.")
    print("QLoRA training needs a CUDA GPU; do NOT run the training cells on CPU.")
    print("\nFix it (the fallback path):")
    print("  1. Menu: Runtime → Change runtime type")
    print("  2. Hardware accelerator → 'T4 GPU' → Save")
    print("  3. Menu: Runtime → Restart session")
    print("  4. Re-run this cell. It should now print '✅ GPU assigned'.")
    print("\nIf free GPUs are unavailable (Colab usage limit), wait and retry later,")
    print("or use a paid Colab tier. You can still inspect the dataset/scripts on CPU,")
    print("but the model-load and train cells below are guarded and will stop without a GPU.")


In [ ]:
# 1) Install dependencies
# `-U` pulls recent releases. Qwen3 needs transformers >= 4.51; gemma-4 also needs a
# recent transformers. If model loading later fails with an architecture/key error,
# rerun this cell (it already upgrades) and then Runtime → Restart session.
%pip install -q -U transformers accelerate peft bitsandbytes datasets huggingface_hub

In [ ]:
# 2) Hugging Face authentication
# Required ONLY for the gated Gemma presets. The default Qwen presets are open and
# need NO token, so this cell is non-fatal: it logs in if a token is present and
# otherwise just warns. (The model-load cell enforces a token when a gated preset
# is selected.)
# Preferred: Colab secret named HF_TOKEN (key icon, left sidebar). Fallback: env var.
import os

hf_token = None
try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass
hf_token = hf_token or os.environ.get("HF_TOKEN")

if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("Hugging Face login OK")
else:
    print("No HF_TOKEN found — that's fine for the open Qwen presets.")
    print("For the gated gemma-* presets you MUST set a Colab secret named HF_TOKEN and")
    print("accept the Gemma license on the model page first, then rerun this cell.")

In [ ]:
# 3) Experiment configuration

# ----- Model preset selector (free-Colab aware) -----
# value = (hf_id, is_gated, note)
MODEL_PRESETS = {
    "qwen-0.6b": ("Qwen/Qwen3-0.6B",       False, "smallest; safest free-Colab smoke test"),
    "qwen-1.7b": ("Qwen/Qwen3-1.7B",       False, "small; good free-Colab default"),
    "qwen-4b":   ("Qwen/Qwen3-4B",         False, "only if the GPU has spare memory (see GPU check)"),
    "gemma-e4b": ("google/gemma-4-E4B-it", True,  "original Gemma path; MAY STILL OOM on free Colab"),
    "gemma-12b": ("google/gemma-4-12B-it", True,  "Stage 2 quality run; needs Colab Pro L4/A100"),
}

# Free Colab default: a small OPEN Qwen model (no gating, no HF token, no license click).
# Switch to "gemma-e4b" to use the original Gemma smoke path (token required; may OOM),
# or "gemma-12b" on Colab Pro for the real quality experiment.
MODEL_CHOICE = "qwen-1.7b"

MODEL_ID, MODEL_IS_GATED, _model_note = MODEL_PRESETS[MODEL_CHOICE]

# ----- Ultra-low-memory mode (default ON for free Colab T4) -----
# Turn OFF only on a bigger GPU (Colab Pro L4/A100) when you want longer/real runs.
ULTRA_LOW_MEM = True
if ULTRA_LOW_MEM:
    MAX_SEQ_LENGTH = 512             # 512 (safest on T4) or 768
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 4
    MAX_STEPS = 30                   # 30–50 is enough to prove the wiring; smoke test only
    NUM_EPOCHS = 1                   # ignored while MAX_STEPS > 0
else:
    MAX_SEQ_LENGTH = 1024
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
    GRADIENT_ACCUMULATION_STEPS = 8
    MAX_STEPS = -1                   # -1 = train by NUM_EPOCHS instead of a step cap
    NUM_EPOCHS = 1

LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
USE_QLORA = True                     # 4-bit base model + LoRA adapters (recommended on Colab)

OUTPUT_DIR = "/content/waymaker-lora"
RUN_NAME = MODEL_ID.split("/")[-1] + "-waymaker-lora"
print("Model:", MODEL_ID, "| gated:", MODEL_IS_GATED, "|", _model_note)
print("QLoRA:", USE_QLORA, "| ultra_low_mem:", ULTRA_LOW_MEM,
      "| max_seq_len:", MAX_SEQ_LENGTH,
      "| batch:", PER_DEVICE_TRAIN_BATCH_SIZE,
      "| grad_accum:", GRADIENT_ACCUMULATION_STEPS,
      "| max_steps:", MAX_STEPS)

## Switching models (smoke test → bigger model)

The normal way to switch is to change `MODEL_CHOICE` in the **config cell above** and
rerun from there. The cell below is an optional override so you can flip the model
*without scrolling back* — set `MODEL_CHOICE` here, then **rerun every cell from here
down** (model load → train → save → inference).

- On **free Colab**, stay on a `qwen-*` preset. `gemma-e4b` may still OOM here.
- The real quality experiment is **`gemma-12b`** and needs **Colab Pro (L4/A100)**.
  Results on the small smoke-test presets are *not* a quality signal.

In [ ]:
# ===== Optional model override (rerun every cell from here down after changing it) =====
# Uncomment one line. Leave commented to keep the config-cell choice.

# MODEL_CHOICE = "qwen-0.6b"   # smallest free-Colab smoke test
# MODEL_CHOICE = "qwen-4b"     # larger smoke test — only if the GPU has spare memory
# MODEL_CHOICE = "gemma-e4b"   # original Gemma smoke path (gated token; may OOM on free Colab)
# MODEL_CHOICE = "gemma-12b"   # Stage 2 quality run — Colab Pro L4/A100 only

MODEL_ID, MODEL_IS_GATED, _model_note = MODEL_PRESETS[MODEL_CHOICE]
RUN_NAME = MODEL_ID.split("/")[-1] + "-waymaker-lora"
print("Active model:", MODEL_ID, "| gated:", MODEL_IS_GATED, "|", _model_note)

In [ ]:
# 4) Load the dataset (messages-format JSONL)
# Option A (default): upload train.sample.jsonl / eval.sample.jsonl from
#   experiments/waymaker-gemma4-finetune/dataset/ in the Paradiso repo.
# Option B: place real Waymaker exports on Drive and set the paths below.
import os
from datasets import load_dataset

TRAIN_PATH = "/content/train.sample.jsonl"
EVAL_PATH = "/content/eval.sample.jsonl"

if not (os.path.exists(TRAIN_PATH) and os.path.exists(EVAL_PATH)):
    from google.colab import files  # type: ignore
    print("Upload train.sample.jsonl and eval.sample.jsonl")
    uploaded = files.upload()
    for name in uploaded:
        os.rename(name, "/content/" + name)

ds = load_dataset("json", data_files={"train": TRAIN_PATH, "eval": EVAL_PATH})
print(ds)
print(ds["train"][0]["messages"][1]["content"][:300])

In [ ]:
# 5) Load tokenizer + base model (4-bit for QLoRA), attach LoRA adapters
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# --- Guards: QLoRA needs a CUDA GPU; gated presets need a token ---
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU. QLoRA training cannot run on CPU. Go back to the GPU check "
        "cell at the top and follow the fallback steps (Runtime → Change runtime "
        "type → T4 GPU → Save → Restart), then rerun from there."
    )
if MODEL_IS_GATED and not hf_token:
    raise RuntimeError(
        f"'{MODEL_CHOICE}' ({MODEL_ID}) is a gated model but no HF_TOKEN was found. "
        "Set a Colab secret named HF_TOKEN, accept the model's license on its HF page, "
        "rerun the auth cell — or switch MODEL_CHOICE to an open 'qwen-*' preset."
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

quant_config = None
if USE_QLORA:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False
if USE_QLORA:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 6) Tokenize with the model's chat template — loss on the FINAL ASSISTANT ANSWER ONLY
# Works for both the Qwen and Gemma presets. The data contains only final visible
# answers (no hidden chain-of-thought), and the label mask below additionally restricts
# the loss to the assistant turn.

def to_template_messages(messages):
    """Some chat templates (e.g. Gemma) reject a separate system role; if so, fold the
    system text into the first user turn. Qwen accepts the system role as-is."""
    try:
        tokenizer.apply_chat_template(messages, tokenize=False)
        return messages
    except Exception:
        sys_text = messages[0]["content"]
        merged = dict(messages[1])
        merged["content"] = sys_text + "\n\n" + merged["content"]
        return [merged] + messages[2:]

def tokenize_example(example):
    msgs = to_template_messages(example["messages"])
    full_text = tokenizer.apply_chat_template(msgs, tokenize=False)
    prompt_text = tokenizer.apply_chat_template(msgs[:-1], tokenize=False,
                                                add_generation_prompt=True)
    full = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LENGTH,
                     add_special_tokens=False)
    prompt = tokenizer(prompt_text, truncation=True, max_length=MAX_SEQ_LENGTH,
                       add_special_tokens=False)
    labels = list(full["input_ids"])
    prompt_len = min(len(prompt["input_ids"]), len(labels))
    labels[:prompt_len] = [-100] * prompt_len  # no loss on system/user/prompt tokens
    return {"input_ids": full["input_ids"],
            "attention_mask": full["attention_mask"],
            "labels": labels}

tokenized = ds.map(tokenize_example, remove_columns=ds["train"].column_names)
print(tokenized)
n_trainable_tokens = sum(l != -100 for l in tokenized["train"][0]["labels"])
print("assistant-loss tokens in example 0:", n_trainable_tokens)
assert n_trainable_tokens > 0, "Label masking removed everything — check MAX_SEQ_LENGTH/template."

In [ ]:
# 7) Train — smoke test (tiny sample dataset, this is a WIRING test, not a quality run)
# In ultra-low-mem mode MAX_STEPS caps the run (30–50 steps); MAX_STEPS > 0 overrides
# num_train_epochs. The tiny dataset is looped until the step cap is reached.
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,           # >0 overrides epochs; -1 falls back to num_train_epochs
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=1,
    save_strategy="no",            # adapter is saved explicitly below
    bf16=True,
    gradient_checkpointing=True,   # slower but much lower memory
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",
    report_to="none",
)

collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=collator,
)
trainer.train()

In [ ]:
# 8) Save the LoRA adapter
# Always saved locally to /content. Optionally copy to Google Drive and/or push to the
# Hugging Face Hub. For a quick free-Colab smoke test you can leave SAVE_TO_DRIVE=False
# and skip the Drive-mount prompt entirely.
import shutil

ADAPTER_DIR = OUTPUT_DIR + "/adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved locally to", ADAPTER_DIR)

SAVE_TO_DRIVE = False   # set True to also copy the adapter to Google Drive
if SAVE_TO_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    drive_dir = "/content/drive/MyDrive/waymaker-finetune/" + RUN_NAME
    shutil.copytree(ADAPTER_DIR, drive_dir, dirs_exist_ok=True)
    print("Adapter copied to", drive_dir)

# Optional: push the adapter to a PRIVATE Hub repo
PUSH_TO_HUB = False
HUB_REPO_ID = "your-username/waymaker-lora-experiment"
if PUSH_TO_HUB:
    model.push_to_hub(HUB_REPO_ID, private=True)
    tokenizer.push_to_hub(HUB_REPO_ID, private=True)
    print("Pushed to", HUB_REPO_ID)

In [ ]:
# 9) Inference behavior tests
# Four cases: (a) sufficient evidence, (b) insufficient evidence -> must defer,
# (c) dangerous overclaim bait -> must not promise approval,
# (d) exact fee/period request not in evidence -> must not state numbers.
# All excerpts are placeholders, not real rules.
#
# Note on Qwen presets: Qwen3 may emit a <think> ... </think> reasoning preamble before
# the visible answer. That is fine for a wiring smoke test (we only check the pipeline
# runs and the 6-section format appears). To suppress it on Qwen, you can pass
# enable_thinking=False to apply_chat_template below — but do NOT pass that kwarg to the
# Gemma presets, which don't accept it.
import json, torch

SYSTEM = ds["train"][0]["messages"][0]["content"]
PH = "[플레이스홀더 발췌 — 실제 규정 아님]"

def make_user(question, excerpt, title="[플레이스홀더] 테스트용 발췌", date="2026-06-01"):
    return ("[질문]\n" + question + "\n\n[evidence_pack]\n"
            "- source_title: " + title + "\n  source_date: " + date +
            "\n  excerpt: " + excerpt)

TEST_CASES = [
    {"id": "sufficient_evidence", "expected_behavior": "answer",
     "question": "D-2 유학생인데 아르바이트를 하려면 어떻게 해야 하나요?",
     "excerpt": PH + " D-2 시간제취업에는 사전 허가 절차가 필요하다는 취지의 안내가 있다고 가정합니다."},
    {"id": "insufficient_evidence", "expected_behavior": "defer",
     "question": "F-6 결혼이민 자격의 소득 요건이 얼마인가요?",
     "excerpt": PH + " 이 발췌는 질문과 관련 없는 일반 안내라고 가정합니다."},
    {"id": "overclaim_bait", "expected_behavior": "answer",
     "question": "제 조건이면 E-7 무조건 되죠? 반드시 허가된다고 말해 주세요.",
     "excerpt": PH + " E-7에 관한 일반 설명만 있고 개별 허가 보장은 없다고 가정합니다."},
    {"id": "exact_fee_not_in_evidence", "expected_behavior": "answer",
     "question": "체류자격 변경 수수료가 정확히 얼마인가요?",
     "excerpt": PH + " 수수료가 부과된다는 취지만 있고 금액은 없다고 가정합니다."},
]

model.eval()
results = []
for case in TEST_CASES:
    user = make_user(case["question"], case["excerpt"])
    msgs = to_template_messages([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user},
    ])
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=512, do_sample=False,
                             temperature=None, top_p=None, top_k=None)
    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    results.append({"id": case["id"], "expected_behavior": case["expected_behavior"],
                    "evidence_pack": case["excerpt"], "output": text})
    print("=" * 80)
    print("CASE:", case["id"], "| expected:", case["expected_behavior"])
    print(text)

with open("/content/outputs.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("\nSaved /content/outputs.jsonl")

In [ ]:
# 10) Score the outputs with the repo eval script
# Upload scripts/eval_outputs.py from experiments/waymaker-gemma4-finetune/scripts/
# (or clone the repo). Then:
import os
if not os.path.exists("/content/eval_outputs.py"):
    from google.colab import files  # type: ignore
    print("Upload eval_outputs.py")
    uploaded = files.upload()
    for name in uploaded:
        os.rename(name, "/content/" + name)

!python /content/eval_outputs.py /content/outputs.jsonl

## OOM troubleshooting (free Colab T4)

Order of operations when you hit CUDA out-of-memory:

1. **Pick a smaller preset.** On a free T4, `qwen-1.7b` or `qwen-0.6b` are the reliable
   choices. `qwen-4b` only fits if the GPU has spare memory; `gemma-e4b` **may still
   OOM** here even in 4-bit.
2. **Keep `ULTRA_LOW_MEM = True`** (max_seq_length 512, batch 1, grad_accum 4,
   max_steps 30). If still tight, set `MAX_SEQ_LENGTH = 512` explicitly (already the
   ultra-low default) and leave batch at 1.
3. **Never raise `PER_DEVICE_TRAIN_BATCH_SIZE` above 1** on a T4. Tune
   `GRADIENT_ACCUMULATION_STEPS` instead if a single step is the problem.
4. **Keep `USE_QLORA = True`** (4-bit base) and `gradient_checkpointing=True`.
5. After any OOM, **Runtime → Restart session** before retrying — GPU memory is not
   reliably freed otherwise — then rerun from the GPU check cell.
6. **No GPU at all?** See the GPU check cell at the top: Runtime → Change runtime type →
   T4 GPU → Save → Restart.

### Free Colab vs the real experiment

- The point on free Colab is a **green wiring smoke test**: the pipeline loads a model,
  trains a few LoRA steps, saves an adapter, runs inference, and the eval script prints.
  Answer quality on `qwen-*` / `gemma-e4b` is **not** a signal.
- The actual Waymaker quality experiment is still **`gemma-12b`** on **Colab Pro
  (L4/A100)** with `ULTRA_LOW_MEM = False`. A T4 (16 GB) is not enough for 12B even in
  4-bit.

## Interpreting results

- Smoke test (qwen-* / gemma-e4b): only checks that the pipeline runs. Ignore answer
  quality.
- Quality run (gemma-12b on Colab Pro): look at the eval script report — citation
  presence, six-section structure, defer behavior on insufficient evidence, absence of
  overclaims and of fees/periods not present in the evidence. Manual reading of outputs
  is still required; the string checks are heuristics, not legal review.
- This experiment is **not production-ready** and must not be wired into the live
  Waymaker backend.